# Notebook Colab (T4) — RAG Formulaire avec Meta Llama 3 8B

Ce notebook utilise **Meta Llama 3 8B Instruct** au lieu de Mistral 7B pour une meilleure compréhension des instructions en français.

## Avantages de Llama 3 8B

- ✅ **Meilleure instruction-following** : Plus fidèle aux prompts
- ✅ **Contexte plus large** : 8192 tokens vs 4096 pour Mistral
- ✅ **Multilingue amélioré** : Meilleur support du français
- ✅ **Raisonnement** : Meilleures capacités de réflexion

Ce notebook permet de :
- Vérifier le GPU disponible et configurer le dépôt.
- Installer les dépendances et construire un petit index.
- Poser des questions avec Llama 3 8B au lieu de Mistral 7B.

> **Astuce :** utilisez un quota réduit de formulaires (ex. 30) pour accélérer l'ingestion sur Colab.

> **Note :** Ce notebook utilise le code intégré directement depuis le dépôt avec toutes les optimisations récentes.

## 1) Vérifier le GPU

In [ ]:
!nvidia-smi

## 2) Préparer le dépôt

- Définissez `RAG_FORM_REPO_URL` si le dépôt n'est pas déjà présent dans `/content/rag-formulaire`.
- Le notebook ajoute automatiquement le dépôt au `PYTHONPATH` pour l'installation en mode développement.

In [ ]:
import os
import pathlib
import sys

REPO_URL = os.environ.get("RAG_FORM_REPO_URL", "").strip()
REPO_URL = "https://github.com/abdelmajidlra/rag-formulaire.git"
WORKDIR = pathlib.Path("/content/rag-formulaire")

if not WORKDIR.exists():
    if not REPO_URL:
        raise ValueError(
            "Définissez RAG_FORM_REPO_URL ou clonez le dépôt dans /content/rag-formulaire avant d'exécuter ce notebook."
        )
    else:
        print(f"Clonage du dépôt depuis {REPO_URL}…")
        get_ipython().system(f"git clone {REPO_URL} {WORKDIR}")

get_ipython().run_line_magic("cd", str(WORKDIR))
if str(WORKDIR) not in sys.path:
    sys.path.append(str(WORKDIR))

# Add the 'src' directory to sys.path for direct module imports
SRC_DIR = WORKDIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

## 3) Installer les dépendances

L'installation en mode développement (`-e .`) permet de modifier le code localement pendant la session Colab.

In [ ]:
get_ipython().system("pip -q install -U pip setuptools wheel")
get_ipython().system("pip -q install -e .")
!pip install -q bitsandbytes

## 4) Paramétrage avec Llama 3 8B

**Configuration spécifique pour Llama 3 8B:**
- Modèle plus grand mais meilleur instruction-following
- Contexte 8K tokens (vs 4K pour Mistral)
- Quantification 4-bit pour tenir sur T4 (15GB VRAM)

Variables d'environnement ajustées:
- `RAG_FORM_GEN_MODEL`: Meta Llama 3 8B Instruct
- `RAG_FORM_MIN_FORMS`: 30 formulaires pour accélérer
- `RAG_FORM_GEN_4BIT`: Activer quantification 4-bit

In [ ]:
from pprint import pprint

os.environ.setdefault("RAG_FORM_BASE_DIR", str(WORKDIR))
os.environ.setdefault("RAG_FORM_MIN_FORMS", "30")
os.environ.setdefault("RAG_FORM_MAX_SYNTH", "0")
os.environ.setdefault("RAG_FORM_ENABLE_GRAPHRAG", "false")

# 🦙 Configuration pour Meta Llama 3 8B Instruct
os.environ["RAG_FORM_GEN_MODEL"] = "meta-llama/Meta-Llama-3-8B-Instruct"
os.environ["RAG_FORM_GEN_4BIT"] = "true"  # Obligatoire pour T4

# Optimisations récentes
os.environ.setdefault("RAG_FORM_CHUNK_SIZE", "400")  # Chunks plus grands
os.environ.setdefault("RAG_FORM_CHUNK_OVERLAP", "80")  # Meilleur contexte
os.environ.setdefault("RAG_FORM_STRICT_VERIFICATION", "false")  # Mode lenient + form code validation

print("Configuration Llama 3 8B en cours :")
pprint({k: os.environ[k] for k in sorted(os.environ) if k.startswith("RAG_FORM_")})

## 5) Construire l'index (BM25 + vecteur)

Cette étape télécharge les formulaires, découpe les documents puis construit les index. Ajustez `min_forms` pour accélérer sur Colab.

> **Optimisations intégrées:**
> - LLM Singleton: Une seule instance du modèle (économise ~50% de mémoire)
> - Chunks 400 tokens: Meilleur contexte que 200 tokens
> - Downloader Deduplication: Évite les doublons dans le manifest
> - Smart Retrieval: Détection automatique des codes de formulaire spécifiques
> - Form Code Validation: Empêche les hallucinations de codes formulaire

In [ ]:
from rag_formulaire.ingest import ingest_pipeline

index_store = ingest_pipeline(min_forms=int(os.environ["RAG_FORM_MIN_FORMS"]))
print(f"Chunks indexés : {len(index_store.chunk_map)}")

### Aperçu du manifest

In [ ]:
import json
from rag_formulaire import config

with open(config.MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print(f"Formulaires disponibles : {len(manifest)}")
for entry in manifest[:3]:
    print(entry)

## 6) Charger le LLM Llama 3 8B

Le Llama 3 8B sera chargé avec le singleton pattern - une seule instance partagée.

**Différences vs Mistral 7B:**
- Plus grande fenêtre de contexte (8K vs 4K)
- Meilleur instruction-following en français
- Légèrement plus lent mais plus précis

In [ ]:
from rag_formulaire import config
from rag_formulaire.evaluation import AdvancedSelfReflector, CRAGEvaluator, verify_response_against_evidence
from rag_formulaire.indexing import load_indexes
from rag_formulaire.llm import LocalLLM
from rag_formulaire.query_processing import AgenticQueryRouter, MultilingualQueryHandler, QueryDecomposer, QueryExpander
from rag_formulaire.reranker import CrossEncoderReranker
from rag_formulaire.retrieval import HybridRetriever

print(f"🦙 Chargement de {config.GEN_MODEL_NAME}...")
print("Ceci peut prendre 1-2 minutes pour la première exécution.")

index_store = load_indexes()
query_handler = MultilingualQueryHandler()
router = AgenticQueryRouter()
expander = QueryExpander()
decomposer = QueryDecomposer()
retriever = HybridRetriever(index_store)
reranker = CrossEncoderReranker()
evaluator = CRAGEvaluator()
reflector = AdvancedSelfReflector()
llm = LocalLLM()  # Singleton - sera réutilisé dans evaluator et reflector

print("✅ Tous les composants chargés avec succès!")

In [ ]:
def ask_question(question: str, evidence_k: int = config.FINAL_EVIDENCE_K):
    """Pipeline RAG complet avec Llama 3 8B."""
    q_orig, q_fr = query_handler.normalize(question)
    route = router.route(q_fr)
    expansions = expander.expand(q_fr, n=3)
    subqueries = decomposer.decompose(q_fr) if route == "MULTI_STEP" else [q_fr]

    candidates = []
    for sub in subqueries:
        for variant in expansions:
            candidates.extend(retriever.retrieve(variant, manifest=None))

    reranked = reranker.rerank(q_fr, candidates, top_n=config.RERANK_TOP_N)
    if not reranked:
        return {"route": route, "answer": "Aucun extrait trouvé.", "evidence": []}

    scores = list(range(len(reranked), 0, -1))
    if not evaluator.is_evidence_strong(scores, reranked):
        return {"route": route, "answer": evaluator.fallback_message(), "evidence": []}

    evidence_texts = [
        f"[{c.base_chunk.form_code}] {c.base_chunk.section_title}: {c.base_chunk.content}"
        for c in reranked[:evidence_k]
    ]
    system_prompt = (
        "Vous êtes un assistant spécialisé dans les formulaires IRCC. Répondez uniquement en français en vous basant sur les "
        "extraits fournis. Citez le code du formulaire et la section."
    )
    user_prompt = q_fr + "Extraits:" + "".join(evidence_texts)
    answer = llm.chat(system_prompt, user_prompt, max_new_tokens=256)

    # Validation des codes de formulaire (empêche hallucinations)
    if verify_response_against_evidence(answer, reranked):
        answer = reflector.reflect(q_fr, answer, reranked)
    else:
        answer = evaluator.fallback_message()

    return {
        "route": route,
        "expansions": expansions,
        "answer": answer,
        "evidence": reranked[:evidence_k],
    }

### Utilitaires d'affichage

Fonction pour afficher les résultats de manière formatée avec Markdown.

In [ ]:
from IPython.display import display, Markdown

def display_result(result, manifest_list=None):
    """
    Affiche la réponse et les sources de manière formatée en Markdown.
    """
    # 1. En-tête avec la route utilisée
    md = f"### 🦙 Réponse Llama 3 8B (Stratégie : `{result['route']}`)\n\n"

    # 2. La réponse générée
    md += f"{result['answer']}\n\n"

    # 3. Les sources (Preuves)
    md += "---\n#### 🔍 Sources utilisées :\n"

    # Création d'un dictionnaire pour retrouver les URL à partir du code formulaire
    url_map = {m['form_code']: m['pdf_url'] for m in manifest_list} if manifest_list else {}

    for i, ev in enumerate(result['evidence'], 1):
        chunk = ev.base_chunk
        form_code = chunk.form_code
        section = chunk.section_title

        # Lien vers le PDF officiel si disponible
        if form_code in url_map:
            source_link = f"[{form_code}]({url_map[form_code]})"
        else:
            source_link = f"**{form_code}**"

        # Petit extrait du texte pour contexte (nettoyé des sauts de ligne)
        preview = chunk.content.replace("\n", " ")[:500] + "..."

        md += f"{i}. {source_link} — *{section}* (Page {chunk.page_number})\n"
        md += f"   > <small>{preview}</small>\n"

    display(Markdown(md))

### Gestion de la mémoire GPU

Outils pour nettoyer le cache GPU entre les requêtes et éviter les erreurs OOM (Out Of Memory).

> **Note:** Llama 3 8B utilise plus de VRAM que Mistral 7B. Le nettoyage est encore plus important.

In [ ]:
import torch
import gc

# 1. Force Python garbage collection
gc.collect()

# 2. Clear CUDA (GPU) cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✅ GPU cache cleared.")

# 3. Check status
!nvidia-smi

## 7) Tests avec questions variées

Testez le pipeline avec une série de questions pour comparer Llama 3 8B vs Mistral 7B.

> **Attendu de Llama 3 8B:**
> - Réponses plus structurées
> - Meilleur respect des instructions (citations de sections)
> - Moins d'hallucinations grâce à meilleur raisonnement
> - Meilleures réponses en français

In [ ]:
import torch
import gc

# Questions de test pour évaluer Llama 3 8B
test_questions = [
    # --- Tests de base ---
    "À quoi sert le formulaire IMM 0008 Annexe 9 ?",
    "Quel est le code du formulaire pour la Déclaration d'union de fait ?",
    
    # --- Test COVID (expansion améliorée) ---
    "Quel formulaire utiliser pour la déclaration d'un parent d'un mineur durant la COVID-19 ?",
    
    # --- Tests de précision ---
    "Qu'est-ce qu'un représentant rémunéré selon le formulaire IMM 5476 ?",
    "Quel formulaire utiliser pour autoriser la communication de renseignements à une personne désignée ?",
    
    # --- Tests complexes ---
    "Quelle est la liste de contrôle des documents pour un permis de travail ?",
    "Comment demander un renvoi des frais de traitement (IMM 5741) ?",
]

print(f"🦙 Lancement de {len(test_questions)} questions avec Llama 3 8B...\n")

for i, question in enumerate(test_questions, 1):
    print(f"▶️ Question {i}/{len(test_questions)}: {question}")

    try:
        # Interrogation du pipeline
        result = ask_question(question)

        # Affichage propre
        display_result(result, manifest_list=manifest)

    except RuntimeError as e:
        if "out of memory" in str(e):
            print("⚠️ ERREUR OOM : Mémoire GPU saturée. Essayez de réduire max_new_tokens.")
            # Nettoyage d'urgence
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        else:
            print(f"⚠️ Erreur inattendue : {e}")

    print("\n" + "="*80 + "\n")

    # --- NETTOYAGE MÉMOIRE CRITIQUE (important pour Llama 8B) ---
    if 'result' in locals():
        del result
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("✅ Tests terminés avec Llama 3 8B!")

## 8) Comparaison Llama 3 8B vs Mistral 7B

### Avantages de Llama 3 8B

| Aspect | Mistral 7B | Llama 3 8B | Gagnant |
|--------|------------|------------|----------|
| **Taille** | 7B params | 8B params | - |
| **Contexte** | 4096 tokens | 8192 tokens | 🦙 |
| **Instruction-following** | Bon | Excellent | 🦙 |
| **Français** | Bon | Très bon | 🦙 |
| **Vitesse** | Rapide | Légèrement plus lent | ⚡ |
| **VRAM (4-bit)** | ~4.5GB | ~5.5GB | ⚡ |
| **Hallucinations** | Occasionnelles | Rares | 🦙 |
| **Citations** | Variables | Meilleures | 🦙 |

### Recommandations

**Utilisez Llama 3 8B si:**
- ✅ Vous avez au moins 6GB de VRAM disponible
- ✅ La qualité prime sur la vitesse
- ✅ Vous avez besoin de réponses structurées
- ✅ Le support français est critique

**Utilisez Mistral 7B si:**
- ✅ VRAM limitée (<5GB)
- ✅ Vitesse critique
- ✅ Réponses courtes suffisantes